# Análisis de Estabilidad Temporal y Regional
## Tesis: Benchmarking Explainable Gradient Boosting and Tabular Deep Learning
## for Predicting Satisfaction with Democracy in Latin America (1995–2024)

**Objetivo:** Responder PI3 y OE3 — ¿los determinantes identificados son
estables entre subperiodos históricos y subregiones geográficas?

Requiere haber ejecutado los notebooks 02 y 04 previamente.

### Estructura
| Sección | Contenido |
|---|---|
| 1–2 | Importaciones y configuración |
| 3 | Correlación Spearman entre rankings SHAP (SP1/SP2/SP3) |
| 4 | Heatmap y bump chart de estabilidad |
| 5 | Análisis de estabilidad por subregión |
| 6 | MAE ordinal por país y subperiodo |
| 7 | Prueba formal de H4 y H5 |
| 8 | Resumen y guardado |

## 1. Importaciones

In [1]:
import sys
sys.path.append("..")

import warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import json

warnings.filterwarnings("ignore")

from utils.config import (
    setup_plots, THEME, PATHS, SUBPERIODOS, SUBREGIONES,
    PAISES_EXCLUIR_TEST, COL_TARGET, COL_PAIS,
    ETIQUETAS, ETIQUETAS_FEATURES, BLOQUES, bloque_de,
)
from utils.io import (
    cargar_pipeline, cargar_split_parquet, cargar_resultados,
    cargar_shap_values, shap_disponible, cargar_mejor_modelo,
)
from utils.plots import (
    plot_heatmap_estabilidad, plot_bump_chart,
    plot_spearman_estabilidad, plot_shap_por_subregion,
    plot_rendimiento_por_pais, save_figure, model_color,
)
from utils.metrics import evaluar
from sklearn.metrics import mean_absolute_error

setup_plots()
print("✓ Importaciones completadas.")

✓ Importaciones completadas.


## 2. Configuración

In [2]:
# =============================================================================
# Parámetros del análisis de estabilidad
# =============================================================================

# ── Modelo para el análisis de estabilidad ────────────────────────────────────
# 'auto' resuelve el mejor modelo desde el notebook 03
MODELO_ESTABILIDAD = "auto"

# ── Visualización de estabilidad temporal ────────────────────────────────────
# 'heatmap'   → figura principal (organizado por bloque temático)
# 'bump_chart'→ figura principal (top-N variables como líneas de ranking)
# 'ambos'     → genera ambas figuras (recomendado para la tesis)
VIZ_ESTABILIDAD = "ambos"

# ── Top N variables para el bump chart ───────────────────────────────────────
TOP_N_BUMP = 15

# ── Métrica para el análisis por país ────────────────────────────────────────
# 'mae_ordinal': robusto, no requiere masa crítica por clase
METRICA_PAIS = "mae_ordinal"

# Resolver modelo
if MODELO_ESTABILIDAD == "auto":
    try:
        sel_path = PATHS["FOLDER_RESULTS"] / "modelo_xai_seleccionado.json"
        if sel_path.exists():
            sel = json.loads(sel_path.read_text())
            MODELO_ESTABILIDAD = sel["modelo_xai"]
        else:
            MODELO_ESTABILIDAD = cargar_mejor_modelo("SP3", "kappa_cuadratico")
    except:
        MODELO_ESTABILIDAD = "XGBoost"

print(f"Modelo para análisis de estabilidad: {MODELO_ESTABILIDAD}")
print(f"Visualización de estabilidad: {VIZ_ESTABILIDAD}")

# Verificar que los SHAP necesarios están disponibles
print("\nDisponibilidad de valores SHAP:")
todo_disponible = True
for sp in ["SP1","SP2","SP3"]:
    disp = shap_disponible(MODELO_ESTABILIDAD, sp)
    print(f"  {MODELO_ESTABILIDAD} — {sp}: {'✓' if disp else '✗ FALTA (ejecutar NB04)'}")
    if not disp:
        todo_disponible = False
if not todo_disponible:
    print("\n⚠ Faltan valores SHAP. Ejecuta el notebook 04 antes de continuar.")

Modelo para análisis de estabilidad: CatBoost
Visualización de estabilidad: ambos

Disponibilidad de valores SHAP:
  CatBoost — SP1: ✗ FALTA (ejecutar NB04)
  CatBoost — SP2: ✗ FALTA (ejecutar NB04)
  CatBoost — SP3: ✓

⚠ Faltan valores SHAP. Ejecuta el notebook 04 antes de continuar.


## 3. Correlación Spearman entre rankings SHAP

La correlación de Spearman entre los rankings de importancia SHAP de
distintos subperiodos mide directamente la estabilidad de los determinantes.

- **r = 1.0**: los mismos factores explican la satisfacción en todos los períodos
- **r < 0.7**: cambio sustancial en los determinantes entre períodos
- **r < 0.5**: cambio estructural (umbral para H5)

In [3]:
# =============================================================================
# Construir tabla de importancias por subperiodo
# =============================================================================
importancias_sp = {}

for sp in ["SP1","SP2","SP3"]:
    try:
        df_shap = cargar_shap_values(MODELO_ESTABILIDAD, sp)
        imp = df_shap.abs().mean()
        importancias_sp[sp] = imp
    except FileNotFoundError:
        print(f"⚠ SHAP no disponible para {MODELO_ESTABILIDAD} — {sp}")

if len(importancias_sp) < 2:
    print("⚠ Se necesitan al menos 2 subperiodos con SHAP para el análisis.")
else:
    # Alinear features comunes
    feats_comunes = list(set.intersection(*[set(v.index) for v in importancias_sp.values()]))
    df_imp = pd.DataFrame({sp: imp[feats_comunes]
                           for sp, imp in importancias_sp.items()})

    # Calcular correlaciones Spearman entre rankings
    df_corr = pd.DataFrame(index=df_imp.columns, columns=df_imp.columns, dtype=float)
    for sp1 in df_imp.columns:
        for sp2 in df_imp.columns:
            r, p = stats.spearmanr(df_imp[sp1].rank(), df_imp[sp2].rank())
            df_corr.loc[sp1, sp2] = r

    print("Correlación Spearman entre rankings SHAP:")
    print(df_corr.round(4).to_string())
    print()

    # Interpretación
    pares = [("SP1","SP2"),("SP1","SP3"),("SP2","SP3")]
    for sp_a, sp_b in pares:
        if sp_a in df_corr.index and sp_b in df_corr.columns:
            r_val = df_corr.loc[sp_a, sp_b]
            if r_val >= 0.8:
                interp = "Alta estabilidad"
            elif r_val >= 0.6:
                interp = "Estabilidad moderada"
            elif r_val >= 0.4:
                interp = "Cambio sustancial"
            else:
                interp = "Cambio estructural"
            print(f"  {sp_a} ↔ {sp_b}: r = {r_val:.4f} → {interp}")

    df_corr.to_csv(PATHS["FOLDER_RESULTS_TABLES"] /
                   f"spearman_rankings_{MODELO_ESTABILIDAD}.csv")

⚠ SHAP no disponible para CatBoost — SP1
⚠ SHAP no disponible para CatBoost — SP2
⚠ Se necesitan al menos 2 subperiodos con SHAP para el análisis.


## 4. Visualizaciones de estabilidad temporal

In [4]:
# =============================================================================
# Heatmap y/o bump chart según VIZ_ESTABILIDAD
# =============================================================================
if 'df_imp' in dir() and not df_imp.empty:

    if VIZ_ESTABILIDAD in ("heatmap", "ambos"):
        # Normalizar importancias para comparabilidad entre subperiodos
        df_imp_norm = df_imp.copy()
        for col in df_imp_norm.columns:
            total = df_imp_norm[col].sum()
            if total > 0:
                df_imp_norm[col] = df_imp_norm[col] / total

        plot_heatmap_estabilidad(
            df_imp_norm,
            metrica="Importancia SHAP normalizada",
            titulo=f"Estabilidad temporal de determinantes — {MODELO_ESTABILIDAD}",
            nombre_archivo=f"05_heatmap_estabilidad_{MODELO_ESTABILIDAD}",
        )

    if VIZ_ESTABILIDAD in ("bump_chart", "ambos"):
        plot_bump_chart(
            df_imp,
            top_n=TOP_N_BUMP,
            titulo=f"Cambio en ranking de importancia SHAP — {MODELO_ESTABILIDAD}",
            nombre_archivo=f"05_bump_chart_{MODELO_ESTABILIDAD}",
        )

    if VIZ_ESTABILIDAD in ("heatmap", "ambos"):
        plot_spearman_estabilidad(
            df_corr.astype(float),
            titulo=f"Correlación Spearman entre rankings SHAP — {MODELO_ESTABILIDAD}",
            nombre_archivo=f"05_spearman_{MODELO_ESTABILIDAD}",
        )

In [5]:
# =============================================================================
# Cambio por bloque temático entre SP1 y SP3
# =============================================================================
if 'df_imp' in dir() and "SP1" in df_imp.columns and "SP3" in df_imp.columns:
    df_cambio = pd.DataFrame({
        "variable": df_imp.index,
        "bloque"  : [bloque_de(v) for v in df_imp.index],
        "etiqueta": [ETIQUETAS_FEATURES.get(v, v) for v in df_imp.index],
        "imp_SP1" : df_imp["SP1"].values,
        "imp_SP3" : df_imp["SP3"].values,
    })
    df_cambio["cambio_absoluto"] = df_cambio["imp_SP3"] - df_cambio["imp_SP1"]
    df_cambio["cambio_pct"]      = (
        (df_cambio["imp_SP3"] - df_cambio["imp_SP1"]) /
        df_cambio["imp_SP1"].replace(0, np.nan)
    ) * 100

    df_cambio_bloque = (
        df_cambio.groupby("bloque")[["imp_SP1","imp_SP3","cambio_absoluto"]]
        .sum().sort_values("cambio_absoluto", ascending=False)
    )
    print("Cambio en importancia SHAP total por bloque (SP1 → SP3):")
    print(df_cambio_bloque.round(4).to_string())

    df_cambio.to_csv(
        PATHS["FOLDER_RESULTS_TABLES"] /
        f"cambio_shap_sp1_sp3_{MODELO_ESTABILIDAD}.csv", index=False)
    print("\n✓ Tabla de cambio guardada")

## 5. Análisis de estabilidad por subregión

In [6]:
# =============================================================================
# SHAP medio por subregión en SP3 (conjunto de prueba)
# =============================================================================
df_te_sp3 = cargar_split_parquet("SP3", "test")

if COL_PAIS not in df_te_sp3.columns:
    print("⚠ Columna pais_nombre no disponible. Usando análisis global.")
else:
    art_ref = cargar_pipeline(MODELO_ESTABILIDAD, "SP3")
    feats_r = art_ref["features"]

    try:
        df_shap_sp3 = cargar_shap_values(MODELO_ESTABILIDAD, "SP3")
        # Asegurar alineación de índices
        df_shap_sp3.index = df_te_sp3.index[:len(df_shap_sp3)]

        shap_por_region = {}
        for sr, paises_sr in SUBREGIONES.items():
            paises_sr_test = [p for p in paises_sr if p not in PAISES_EXCLUIR_TEST]
            mask = df_te_sp3[COL_PAIS].isin(paises_sr_test)
            if mask.sum() < 20:
                continue
            idx_sr = df_te_sp3[mask].index
            idx_sr_shap = [i for i in range(len(df_te_sp3))
                           if df_te_sp3.index[i] in idx_sr]
            if idx_sr_shap:
                shap_sr = df_shap_sp3.iloc[idx_sr_shap]
                shap_por_region[sr] = shap_sr.abs().mean()

        if shap_por_region:
            df_shap_region = pd.DataFrame(shap_por_region)
            plot_shap_por_subregion(
                df_shap_region,
                top_n=10,
                titulo=f"Importancia SHAP por subregión — {MODELO_ESTABILIDAD} · SP3",
                nombre_archivo=f"05_shap_subregion_{MODELO_ESTABILIDAD}",
            )

            # Correlación de rankings entre subregiones
            print("\nCorrelación Spearman entre rankings por subregión:")
            srs = list(df_shap_region.columns)
            corr_sr = pd.DataFrame(index=srs, columns=srs, dtype=float)
            for sr1 in srs:
                for sr2 in srs:
                    r, _ = stats.spearmanr(
                        df_shap_region[sr1].rank(),
                        df_shap_region[sr2].rank()
                    )
                    corr_sr.loc[sr1, sr2] = r
            print(corr_sr.round(3).to_string())
            corr_sr.to_csv(
                PATHS["FOLDER_RESULTS_TABLES"] /
                f"spearman_subregion_{MODELO_ESTABILIDAD}.csv")

    except FileNotFoundError as e:
        print(f"⚠ {e}")

⚠ Columna pais_nombre no disponible. Usando análisis global.


## 6. MAE ordinal por país y subperiodo

In [ ]:
# =============================================================================
# Análisis de rendimiento por país para los 3 subperiodos
# Métrica: MAE ordinal (robusta ante clases minoritarias)
# =============================================================================
import joblib

filas_mae = []

for sp in ["SP1","SP2","SP3"]:
    df_te = cargar_split_parquet(sp, "test")
    if COL_PAIS not in df_te.columns:
        continue

    y_te = df_te[COL_TARGET].astype(int).values

    for modelo in ["OLO","XGBoost","CatBoost","LightGBM","TabNet"]:
        try:
            art  = cargar_pipeline(modelo, sp)
            tipo = art["tipo_modelo"]
            feats_m = art["features"]
            X_te = df_te[[f for f in feats_m if f in df_te.columns]]
            X_te = X_te.reindex(columns=feats_m)

            if tipo in ("olo", "tabnet"):
                vars_cat_m = [c for c in art.get("vars_categoricas", []) if c in X_te.columns]
                cols_num_m = [c for c in feats_m if c not in vars_cat_m]
                X_num_m = pd.DataFrame(art["imp_num"].transform(X_te[cols_num_m]),
                                    columns=cols_num_m, index=X_te.index)
                if vars_cat_m and art.get("imp_cat") is not None:
                    X_cat_m = pd.DataFrame(art["imp_cat"].transform(X_te[vars_cat_m]),
                                        columns=vars_cat_m, index=X_te.index)
                    X_imp = pd.concat([X_num_m, X_cat_m], axis=1)[feats_m]
                else:
                    X_imp = X_num_m
                cols_sc_m = [c for c in feats_m if c not in vars_cat_m]
                X_sc = X_imp.copy()
                X_sc[cols_sc_m] = art["scaler"].transform(X_imp[cols_sc_m])
                y_raw = art["modelo"].predict(X_sc.values if tipo == "olo"
                            else X_sc.values.astype(np.float32))
            else:
                X_in = X_te.copy()
                if modelo == "CatBoost":
                    for col in art.get("vars_categoricas",[]):
                        if col in X_in.columns:
                            X_in[col] = X_in[col].fillna(-999).astype(int).astype(str)
                y_raw = art["modelo"].predict(X_in)

            y_pred = y_raw.flatten() if hasattr(y_raw,"flatten") else y_raw

            # MAE por país
            for pais in df_te[COL_PAIS].unique():
                if pais in PAISES_EXCLUIR_TEST:
                    continue
                mask = df_te[COL_PAIS] == pais
                if mask.sum() < 10:
                    continue
                mae = mean_absolute_error(y_te[mask], y_pred[mask])
                sr  = next((s for s, ps in SUBREGIONES.items() if pais in ps),
                           "Sin clasificar")
                filas_mae.append({
                    "modelo":modelo,"subperiodo":sp,"pais":pais,
                    "subregion":sr,"mae_ordinal":mae,"n":int(mask.sum()),
                })
        except FileNotFoundError:
            pass
        except Exception as e:
            print(f"  ⚠ {modelo} {sp}: {e}")

df_mae_completo = pd.DataFrame(filas_mae)
if not df_mae_completo.empty:
    print("MAE Ordinal promedio por modelo y subperiodo:")
    pivot_mae = df_mae_completo.groupby(["modelo","subperiodo"])["mae_ordinal"].mean().unstack()
    print(pivot_mae.round(4).to_string())
    df_mae_completo.to_csv(
        PATHS["FOLDER_RESULTS_TABLES"] / "mae_por_pais_todos.csv", index=False)
    print("\n✓ Tabla MAE completa guardada")

In [8]:
# Heatmap: MAE por subregión × subperiodo para el modelo principal
if not df_mae_completo.empty:
    df_sr_sp = (df_mae_completo[df_mae_completo["modelo"] == MODELO_ESTABILIDAD]
                .groupby(["subregion","subperiodo"])["mae_ordinal"].mean()
                .unstack())

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(df_sr_sp, annot=True, fmt=".3f", cmap="YlOrRd",
                linewidths=0.3, ax=ax,
                cbar_kws={"label": "MAE Ordinal"})
    ax.set_title(f"MAE Ordinal por subregión y subperiodo — {MODELO_ESTABILIDAD}",
                 fontweight="bold")
    save_figure(f"05_mae_subregion_subperiodo_{MODELO_ESTABILIDAD}")
    plt.show()

## 7. Prueba formal de H4 y H5

**H4:** la importancia de las variables varía significativamente entre
subperiodos, con seguridad ciudadana y corrupción ganando peso en SP2 y SP3.

**H5:** la correlación Spearman SP1-SP3 será significativamente menor que SP1-SP2,
evidenciando cambio estructural post-pandemia.

In [9]:
# =============================================================================
# Prueba de H4: ¿Qué bloques cambiaron más entre SP1 y SP3?
# =============================================================================
if 'df_cambio' in dir() and not df_cambio.empty:
    print("=" * 60)
    print("PRUEBA DE H4")
    print("=" * 60)
    print("Cambio en importancia SHAP por bloque (SP1 → SP3):")
    print()

    pred_h4 = {
        "Corrupción y seguridad"           : "↑ esperado",
        "Características sociodemográficas": "↑ esperado",
        "Confianza institucional"          : "→ estable",
        "Evaluación económica"             : "→ estable",
        "Percepción política"              : "↑ posible",
    }

    for bloque, row in df_cambio_bloque.iterrows():
        pred = pred_h4.get(bloque, "—")
        direccion = "↑" if row["cambio_absoluto"] > 0 else "↓"
        confirmado = ""
        if pred == "↑ esperado" and row["cambio_absoluto"] > 0:
            confirmado = " ✓ Confirma H4"
        elif pred == "↑ esperado" and row["cambio_absoluto"] <= 0:
            confirmado = " ✗ No confirma H4"
        print(f"  {bloque:<42}: {direccion} ({row['cambio_absoluto']:+.4f})  "
              f"Predicción: {pred}{confirmado}")

print()
print("=" * 60)
print("PRUEBA DE H5")
print("=" * 60)

if 'df_corr' in dir():
    if "SP1" in df_corr.index and "SP2" in df_corr.columns and "SP3" in df_corr.columns:
        r_sp1_sp2 = float(df_corr.loc["SP1","SP2"])
        r_sp1_sp3 = float(df_corr.loc["SP1","SP3"])
        diferencia = r_sp1_sp2 - r_sp1_sp3

        print(f"  r(SP1, SP2) = {r_sp1_sp2:.4f}")
        print(f"  r(SP1, SP3) = {r_sp1_sp3:.4f}")
        print(f"  Diferencia  = {diferencia:+.4f}")
        print()
        if r_sp1_sp3 < r_sp1_sp2:
            print("  ✓ H5 CONFIRMADA: la correlación SP1-SP3 es menor que SP1-SP2,")
            print("    indicando cambio estructural post-pandémico en los determinantes.")
        else:
            print("  ✗ H5 NO CONFIRMADA: la correlación SP1-SP3 no es menor que SP1-SP2.")
        if diferencia > 0.15:
            print(f"    La diferencia ({diferencia:.3f}) sugiere cambio estructural sustancial.")
        elif diferencia > 0.05:
            print(f"    La diferencia ({diferencia:.3f}) es moderada.")
        else:
            print(f"    La diferencia ({diferencia:.3f}) es pequeña.")


PRUEBA DE H5


## 8. Resumen y guardado

In [10]:
print("=" * 60)
print("RESUMEN — Análisis de estabilidad temporal y regional")
print("=" * 60)
print(f"  Modelo analizado : {MODELO_ESTABILIDAD}")
print(f"  Subperiodos      : SP1, SP2, SP3")
print()
print("Archivos generados:")
for f in sorted(PATHS["FOLDER_RESULTS_FIGURES"].glob("05_*.png")):
    print(f"  {f.name}")
for f in sorted(PATHS["FOLDER_RESULTS_TABLES"].glob("*spearman*")):
    print(f"  {f.name}")
for f in sorted(PATHS["FOLDER_RESULTS_TABLES"].glob("*mae*")):
    print(f"  {f.name}")

RESUMEN — Análisis de estabilidad temporal y regional
  Modelo analizado : CatBoost
  Subperiodos      : SP1, SP2, SP3

Archivos generados:
